# GeoHab 2026 — Combined Pipeline

## Recipe

| Component | Weight | Detail |
|---|---|---|
| **Our LGBM** | 0.45 | 26 simple features, Optuna-tuned params, seed averaging [52,62,72], verde BlockKFold 150m |
| **CNN-v2 ensemble** | 0.55 | ResNet18 + EfficientNet-B0 + ConvNeXt-Tiny, 96×96 px, Path B (pre-computed) |
| **E7 Bayesian prior adj.** | post-hoc | TEST_PRIOR / TRAIN_PRIOR per class, renorm, argmax |

**CV convention:** OOF CV always pre-Bayes. Bayesian adjustment applied at test time only.

**Fold alignment:** Both LGBM and CNN use identical `verde.BlockKFold(spacing=150, n_splits=5, shuffle=True, random_state=42)` — required for blend OOF CV to be meaningful.

## External Resources & Acknowledgements

This solution incorporates publicly available CNN-based out-of-fold (OOF) predictions shared by Matteo (competition participant). These predictions were used as an additional feature alongside the tabular and geospatial features developed in this notebook.
The validation strategy (BlockKFold) and Bayesian prior adjustment were adapted from publicly shared work by Matteo as well. You can check it out [here](https://www.kaggle.com/code/mycarta/geohab-2026-solution-write-up)

All other feature engineering, model development, and ensembling were implemented independently.

## The final model combines:
- Multibeam-derived features (depth, backscatter)
- Spatial grid-based features
- CNN-based OOF predictions

## 1  Setup & Installs

In [1]:
import os, glob, pathlib, warnings
warnings.filterwarnings('ignore')
os.system('pip install -q verde timm')

import numpy as np
import pandas as pd
import rasterio
import verde as vd
import lightgbm as lgb
from sklearn.metrics import f1_score, classification_report
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

IS_KAGGLE = os.path.exists('/kaggle/input')
print(f'Running on Kaggle: {IS_KAGGLE}')
print(f'verde {vd.__version__}  |  lightgbm {lgb.__version__}')

# ── CONFIG ────────────────────────────────────────────────────────────────────
CLASS_NAMES = ['ALG', 'FMAT', 'NVB', 'SGAM', 'SGZ']
LABEL_MAP   = {c: i for i, c in enumerate(CLASS_NAMES)}
NUM_CLASSES = len(CLASS_NAMES)

# Verde BlockKFold — shared by LGBM and CNN
BKF_SPACING  = 150
BKF_N_SPLITS = 5
BKF_SEED     = 42

# Blend weights
BLEND_W_TAB = 0.45
BLEND_W_CNN = 0.55

# E7 Bayesian prior adjustment — Ierodiaconou et al. (2018) Fig 7
# Order: ALG, FMAT, NVB, SGAM, SGZ
TRAIN_PRIOR = np.array([0.1090, 0.2468, 0.4853, 0.0272, 0.1317])
TEST_PRIOR  = np.array([0.2960, 0.1730, 0.3670, 0.0610, 0.1020])
BAYES_RATIO = TEST_PRIOR / (TRAIN_PRIOR + 1e-12)

# Our Optuna-tuned LGBM params
LGB_PARAMS = dict(
    n_estimators      = 2000,
    learning_rate     = 0.07918500203202344,
    max_depth         = 11,
    num_leaves        = 77,
    subsample         = 0.8055152296793877,
    colsample_bytree  = 0.7103689169741864,
    reg_alpha         = 0.014069050212386091,
    reg_lambda        = 0.7078441464734376,
    min_child_samples = 9,
    class_weight      = 'balanced',
    device            = 'gpu',
    n_jobs            = -1,
    verbose           = -1,
)

SEEDS = [52, 62, 72]

# Path helper
def find_file(filename, search_root=None):
    if search_root is None:
        search_root = '/kaggle' if IS_KAGGLE else '.'
    matches = glob.glob(f'{search_root}/**/{filename}', recursive=True)
    if not matches:
        raise FileNotFoundError(f'{filename!r} not found under {search_root!r}')
    return str(pathlib.Path(matches[0]).parent)

COMP_DIR   = find_file('train.csv')
SAMPLE_DIR = find_file('sample_submission.csv')
print(f'Competition data: {COMP_DIR}')
print('Config OK ✓')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.6/188.6 kB 9.1 MB/s eta 0:00:00
Running on Kaggle: True
verde v1.9.0  |  lightgbm 4.6.0
Competition data: /kaggle/input/competitions/geohab-mlwg-competition-2026
Config OK ✓


## 2  Data Loading

In [2]:
back     = rasterio.open(f'{COMP_DIR}/MBES/backscatter.tif')
bath     = rasterio.open(f'{COMP_DIR}/MBES/bathymetry.tif')
train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
test_df  = pd.read_csv(f'{COMP_DIR}/test.csv')

back_data = back.read(1)
bath_data = bath.read(1)

assert len(train_df) == 6256
assert len(test_df)  == 98

train_labels_str = train_df['class'].to_numpy()
train_labels_int = np.array([LABEL_MAP[c] for c in train_labels_str])
train_coords     = train_df[['x', 'y']].to_numpy()
n_pts            = len(train_df)
tab_test_ids     = test_df['ID'].to_numpy()

print(f'Train: {len(train_df)}  Test: {len(test_df)}')
print('\nClass distribution:')
for cls, n in pd.Series(train_labels_str).value_counts().reindex(CLASS_NAMES).items():
    print(f'  {cls}: {n:4d}  ({100*n/n_pts:.1f}%)')

Train: 6256  Test: 98

Class distribution:
  ALG:  682  (10.9%)
  FMAT: 1544  (24.7%)
  NVB: 3036  (48.5%)
  SGAM:  170  (2.7%)
  SGZ:  824  (13.2%)


## 3  Simple Feature Extraction
26 features: 3×3 grid at ±12px (18 grid features) + 8 basic neighbourhood stats.

In [3]:
def get_neighborhood_features(x, y):
    row, col = bath.index(x, y)
    features = []
    for dr in [-12, 0, 12]:
        for dc in [-12, 0, 12]:
            r, c = row + dr, col + dc
            try:
                depth   = bath_data[r, c]
                scatter = back_data[r, c]
            except:
                depth   = 0
                scatter = 0
            features.append(depth)
            features.append(scatter)
    return features

def get_advanced_features(x, y):
    row, col = bath.index(x, y)
    depths, scatters = [], []
    for dr in [-10, 0, 10]:
        for dc in [-10, 0, 10]:
            r, c = row + dr, col + dc
            if 0 <= r < bath_data.shape[0] and 0 <= c < bath_data.shape[1]:
                depths.append(bath_data[r, c])
                scatters.append(back_data[r, c])
            else:
                depths.append(0)
                scatters.append(0)
    return [
        np.mean(depths),   np.std(depths),
        np.min(depths),    np.max(depths),
        np.mean(scatters), np.std(scatters),
        np.min(scatters),  np.max(scatters),
    ]

print('Extracting features...')
feats = train_df.apply(lambda r: get_neighborhood_features(r['x'], r['y']), axis=1)
train_feat_cols = [f'f{i}' for i in range(len(feats.iloc[0]))]
train_df[train_feat_cols] = pd.DataFrame(feats.tolist(), index=train_df.index)

adv = train_df.apply(lambda r: get_advanced_features(r['x'], r['y']), axis=1)
adv_cols = ['depth_mean','depth_std','depth_min','depth_max',
            'scatter_mean','scatter_std','scatter_min','scatter_max']
train_df[adv_cols] = pd.DataFrame(adv.tolist(), index=train_df.index)

feats = test_df.apply(lambda r: get_neighborhood_features(r['x'], r['y']), axis=1)
test_df[train_feat_cols] = pd.DataFrame(feats.tolist(), index=test_df.index)

adv = test_df.apply(lambda r: get_advanced_features(r['x'], r['y']), axis=1)
test_df[adv_cols] = pd.DataFrame(adv.tolist(), index=test_df.index)

FEATURES = train_feat_cols + adv_cols
X_tab      = train_df[FEATURES].to_numpy()
X_tab_test = test_df[FEATURES].to_numpy()
print(f'Features: {len(FEATURES)}  |  X_tab: {X_tab.shape}  |  X_tab_test: {X_tab_test.shape}')

Extracting features...
Features: 26  |  X_tab: (6256, 26)  |  X_tab_test: (98, 26)


## 4  Shared verde BlockKFold

Identical splitter used by both LGBM and CNN. Fold assignments must match for blend OOF CV to be meaningful.

In [4]:
kfold  = vd.BlockKFold(
    spacing      = BKF_SPACING,
    n_splits     = BKF_N_SPLITS,
    shuffle      = True,
    random_state = BKF_SEED,
)
splits = list(kfold.split(train_coords))

point_folds = np.full(n_pts, -1, dtype=int)
for fold_id, (_, va_idx) in enumerate(splits):
    point_folds[va_idx] = fold_id
assert (point_folds >= 0).all()
unique_folds = np.sort(np.unique(point_folds))

print(f'verde.BlockKFold: spacing={BKF_SPACING}m, n_splits={BKF_N_SPLITS}, seed={BKF_SEED}')
print('Fold sizes (train / val):')
for i, (tr, va) in enumerate(splits):
    print(f'  Fold {i+1}: train={len(tr):4d}  val={len(va):4d}')

verde.BlockKFold: spacing=150m, n_splits=5, seed=42
Fold sizes (train / val):
  Fold 1: train=5362  val= 894
  Fold 2: train=4727  val=1529
  Fold 3: train=5135  val=1121
  Fold 4: train=5021  val=1235
  Fold 5: train=4779  val=1477


## 5  LGBM Tabular Component — Seed Averaging

Our Optuna-tuned LightGBM with seed averaging over [52, 62, 72].
Same verde folds used by CNN so OOF blend CV is valid.

In [5]:
tab_oof        = np.zeros((n_pts,            NUM_CLASSES), dtype=np.float64)
tab_test_proba = np.zeros((len(X_tab_test),  NUM_CLASSES), dtype=np.float64)
seed_scores    = []

for seed in SEEDS:
    print(f'\n===== Seed {seed} =====')
    params = {**LGB_PARAMS, 'random_state': seed}

    seed_oof        = np.zeros((n_pts,           NUM_CLASSES), dtype=np.float64)
    seed_test_proba = np.zeros((len(X_tab_test), NUM_CLASSES), dtype=np.float64)

    for fold_id, (tr_idx, va_idx) in enumerate(splits):
        X_tr, y_tr = X_tab[tr_idx], train_labels_int[tr_idx]
        X_va, y_va = X_tab[va_idx], train_labels_int[va_idx]

        model = LGBMClassifier(**params)
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            callbacks=[early_stopping(50, verbose=False), log_evaluation(0)]
        )

        seed_oof[va_idx]  = model.predict_proba(X_va)
        seed_test_proba  += model.predict_proba(X_tab_test) / len(splits)

        fold_f1 = f1_score(y_va, seed_oof[va_idx].argmax(1), average='weighted')
        print(f'  Fold {fold_id+1}  val={len(va_idx):4d}  F1={fold_f1:.4f}')

    seed_f1 = f1_score(train_labels_int, seed_oof.argmax(1), average='weighted')
    seed_scores.append(seed_f1)
    print(f'  Seed {seed} OOF F1: {seed_f1:.5f}')

    tab_oof        += seed_oof        / len(SEEDS)
    tab_test_proba += seed_test_proba / len(SEEDS)

assert np.allclose(tab_oof.sum(1),        1.0, atol=1e-5)
assert np.allclose(tab_test_proba.sum(1), 1.0, atol=1e-5)

tab_fold_f1s = np.array([
    f1_score(train_labels_int[point_folds == f],
             tab_oof[point_folds == f].argmax(1), average='weighted')
    for f in unique_folds
])
tab_cv_mean = float(tab_fold_f1s.mean())
tab_cv_se   = float(tab_fold_f1s.std(ddof=1) / np.sqrt(len(unique_folds)))

print('\n==============================')
print('Seed Scores:', [round(s,5) for s in seed_scores])
print(f'Tabular OOF CV (pre-Bayes): {tab_cv_mean:.4f} +/- SE {tab_cv_se:.4f}')
print(f'Per-fold: {tab_fold_f1s.round(4).tolist()}')
print('==============================')


===== Seed 52 =====


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  Fold 1  val= 894  F1=0.5901
  Fold 2  val=1529  F1=0.6482
  Fold 3  val=1121  F1=0.6171
  Fold 4  val=1235  F1=0.6885
  Fold 5  val=1477  F1=0.6286
  Seed 52 OOF F1: 0.63493

===== Seed 62 =====
  Fold 1  val= 894  F1=0.5922
  Fold 2  val=1529  F1=0.6397
  Fold 3  val=1121  F1=0.6112
  Fold 4  val=1235  F1=0.6949
  Fold 5  val=1477  F1=0.6624
  Seed 62 OOF F1: 0.63984

===== Seed 72 =====
  Fold 1  val= 894  F1=0.5949
  Fold 2  val=1529  F1=0.6347
  Fold 3  val=1121  F1=0.6056
  Fold 4  val=1235  F1=0.6988
  Fold 5  val=1477  F1=0.5970
  Seed 72 OOF F1: 0.62732

Seed Scores: [0.63493, 0.63984, 0.62732]
Tabular OOF CV (pre-Bayes): 0.6350 +/- SE 0.0177
Per-fold: [0.5959, 0.6426, 0.6038, 0.6956, 0.637]


## 6  CNN-v2 Ensemble — Path B (load pre-computed outputs)

Loads `ensemble_outputs.npz` from the attached CNN dataset.
Contains:
- `oof`       — (6256, 5) ensemble OOF probabilities (last-epoch, pre-Bayes)
- `test_raw`  — (98, 5) ensemble test probabilities (pre-Bayes)

Path A (train from scratch) is preserved in the original solution notebook for reference.

In [6]:
npz_dir = find_file('ensemble_outputs.npz')
arr     = np.load(f'{npz_dir}/ensemble_outputs.npz')

cnn_oof        = arr['oof'].astype(np.float64)       # (6256, 5)
cnn_test_proba = arr['test_raw'].astype(np.float64)  # (98, 5)

assert cnn_oof.shape        == (6256, NUM_CLASSES)
assert cnn_test_proba.shape == (98,   NUM_CLASSES)
assert np.allclose(cnn_oof.sum(1),        1.0, atol=1e-5)
assert np.allclose(cnn_test_proba.sum(1), 1.0, atol=1e-5)

print(f'cnn_oof:        {cnn_oof.shape}  sums-to-1 OK')
print(f'cnn_test_proba: {cnn_test_proba.shape}  sums-to-1 OK')

# CNN OOF CV (pre-Bayes)
cnn_fold_f1s = np.array([
    f1_score(train_labels_int[point_folds == f],
             cnn_oof[point_folds == f].argmax(1), average='weighted')
    for f in unique_folds
])
cnn_cv_mean = float(cnn_fold_f1s.mean())
cnn_cv_se   = float(cnn_fold_f1s.std(ddof=1) / np.sqrt(len(unique_folds)))
print(f'CNN OOF CV (pre-Bayes):     {cnn_cv_mean:.4f} +/- SE {cnn_cv_se:.4f}')

cnn_oof:        (6256, 5)  sums-to-1 OK
cnn_test_proba: (98, 5)  sums-to-1 OK
CNN OOF CV (pre-Bayes):     0.7254 +/- SE 0.0404


## 7  Blend + E7 Bayesian Prior Adjustment + Submission

In [7]:
# ── OOF blend CV (pre-Bayes) ──────────────────────────────────────────────────
blend_oof = BLEND_W_TAB * tab_oof + BLEND_W_CNN * cnn_oof
assert np.allclose(blend_oof.sum(1), 1.0, atol=1e-5)

blend_fold_f1s = np.array([
    f1_score(train_labels_int[point_folds == f],
             blend_oof[point_folds == f].argmax(1), average='weighted')
    for f in unique_folds
])
blend_cv_mean = float(blend_fold_f1s.mean())
blend_cv_se   = float(blend_fold_f1s.std(ddof=1) / np.sqrt(len(unique_folds)))

print(f'Tabular OOF CV  (pre-Bayes): {tab_cv_mean:.4f} +/- SE {tab_cv_se:.4f}')
print(f'CNN OOF CV      (pre-Bayes): {cnn_cv_mean:.4f} +/- SE {cnn_cv_se:.4f}')
print(f'Blend OOF CV    (pre-Bayes): {blend_cv_mean:.4f} +/- SE {blend_cv_se:.4f}')
print(f'Per-fold blend: {blend_fold_f1s.round(4).tolist()}')

print('\nBlend OOF per-class F1 (pre-Bayes):')
print(classification_report(train_labels_int, blend_oof.argmax(1),
                             target_names=CLASS_NAMES, digits=4))

Tabular OOF CV  (pre-Bayes): 0.6350 +/- SE 0.0177
CNN OOF CV      (pre-Bayes): 0.7254 +/- SE 0.0404
Blend OOF CV    (pre-Bayes): 0.7490 +/- SE 0.0242
Per-fold blend: [0.8367, 0.6895, 0.7368, 0.7503, 0.732]

Blend OOF per-class F1 (pre-Bayes):
              precision    recall  f1-score   support

         ALG     0.6154    0.7507    0.6764       682
        FMAT     0.7155    0.6237    0.6664      1544
         NVB     0.8232    0.9262    0.8717      3036
        SGAM     0.4292    0.5706    0.4899       170
         SGZ     0.7798    0.4126    0.5397       824

    accuracy                         0.7551      6256
   macro avg     0.6726    0.6568    0.6488      6256
weighted avg     0.7575    0.7551    0.7456      6256



In [8]:
# ── Test blend + E7 Bayesian prior adjustment ─────────────────────────────────
blend_raw = BLEND_W_TAB * tab_test_proba + BLEND_W_CNN * cnn_test_proba
assert np.allclose(blend_raw.sum(1), 1.0, atol=1e-5)

print('Bayesian prior adjustment ratios (TEST_PRIOR / TRAIN_PRIOR):')
for c, r in zip(CLASS_NAMES, BAYES_RATIO):
    print(f'  {c}: x{r:.4f}')

blend_adj  = blend_raw * BAYES_RATIO[np.newaxis, :]
blend_adj /= blend_adj.sum(axis=1, keepdims=True)
assert np.allclose(blend_adj.sum(1), 1.0, atol=1e-6)

preds_int = blend_adj.argmax(axis=1)
preds_str = [CLASS_NAMES[p] for p in preds_int]

print('\nTest prediction class distribution (post-Bayes):')
for cls, cnt in pd.Series(preds_str).value_counts().reindex(CLASS_NAMES, fill_value=0).items():
    print(f'  {cls}: {cnt}')

# Pre-Bayes distribution for comparison
pre_str = [CLASS_NAMES[p] for p in blend_raw.argmax(1)]
print('\nTest prediction class distribution (pre-Bayes):')
for cls, cnt in pd.Series(pre_str).value_counts().reindex(CLASS_NAMES, fill_value=0).items():
    print(f'  {cls}: {cnt}')

# Bayes flip count
flips = sum(a != b for a, b in zip(pre_str, preds_str))
print(f'\nBayes flipped {flips}/98 predictions')

Bayesian prior adjustment ratios (TEST_PRIOR / TRAIN_PRIOR):
  ALG: x2.7156
  FMAT: x0.7010
  NVB: x0.7562
  SGAM: x2.2426
  SGZ: x0.7745

Test prediction class distribution (post-Bayes):
  ALG: 28
  FMAT: 15
  NVB: 39
  SGAM: 10
  SGZ: 6

Test prediction class distribution (pre-Bayes):
  ALG: 22
  FMAT: 20
  NVB: 41
  SGAM: 9
  SGZ: 6

Bayes flipped 7/98 predictions


In [9]:
# ── Build and save submission ─────────────────────────────────────────────────
sample_sub = pd.read_csv(f'{SAMPLE_DIR}/sample_submission.csv')

sub_df = (pd.DataFrame({'ID': tab_test_ids, 'class': preds_str})
          .set_index('ID')
          .reindex(sample_sub['ID'].values)
          .reset_index())

assert len(sub_df) == 98
assert sub_df['class'].isna().sum() == 0

out_path = '/kaggle/working/submission.csv' 
sub_df.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(f'\nBlend OOF CV (pre-Bayes): {blend_cv_mean:.4f} +/- SE {blend_cv_se:.4f}')
print(f'  Tabular component:      {tab_cv_mean:.4f} +/- SE {tab_cv_se:.4f}')
print(f'  CNN component:          {cnn_cv_mean:.4f} +/- SE {cnn_cv_se:.4f}')

Saved: /kaggle/working/submission.csv

Blend OOF CV (pre-Bayes): 0.7490 +/- SE 0.0242
  Tabular component:      0.6350 +/- SE 0.0177
  CNN component:          0.7254 +/- SE 0.0404


## Validation & Generalization

While the solution achieved a top rank on the public leaderboard, a drop in private leaderboard performance was observed. This indicates some degree of overfitting to the public split.

Potential reasons include:
- Sensitivity to spatial distribution differences between public and private sets
- Heavy reliance on spatial features (e.g., grid-based features)
- Incorporation of CNN-based predictions that may generalize differently across regions

To mitigate this, future improvements could include:
- More robust spatial cross-validation strategies
- Stronger regularization
- Improved ensemble diversity

Other experiments of mine regarding this competition are on GitHub [here](https://github.com/CheeseCakeyy/GeoHab-2026-MLWG-Competition/tree/main/Experiments)

This competition was a fun ride — from learning what geospatial data is to building a full pipeline. 
Also got to interact with some amazing people along the way, which was honestly the biggest win xD.
Definitely looking forward to exploring this space more, this feels like just the start.